In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

### Hyperparameters

In [2]:
device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'    

batch_size = 16
context_length = 32
n_embd = 64
dropout = 0.0
n_head = 4
n_layer = 4

eval_iters = 200
max_iters = 5000
eval_interval = 100

### Preparing Data
- Read Corpus, 

- Create Vocabularies(Tokens)

- Create Encoding and Decoding functions (Tokenizers)

- Split Data

- Create Function to make batches of tokenized data 

In [3]:
with open('input.txt', 'r') as f:
    text = f.read()

print(f'{len(text) = }')

len(text) = 1115394


In [4]:
vocabs = sorted(list(set(text)))
vocab_size = len(vocabs)
print(f'{vocab_size= }')

stoi = {s:i for i,s in enumerate(vocabs)}
itos = {i:s for s,i in stoi.items()}
encode = lambda txt: [stoi[c] for c in txt]
decode = lambda ids: [itos[i] for i in ids] 

vocab_size= 65


In [5]:
data = encode(text)
print(f'{len(data) = }')

len(data) = 1115394


In [6]:
data = torch.tensor(data=data, dtype=torch.long)
train_data = data[:int(0.9*len(data))]
val_data = data[int(0.9*len(data)):]
print(f'{len(train_data)}\n{len(val_data)}')

1003854
111540


In [8]:
train_data

tensor([18, 47, 56,  ..., 43, 56, 43])

In [14]:
idx = torch.randint(0, len(data)-16-1, (4,))
print(idx)
[data[i:i+16] for i in idx]
torch.stack([data[i:i+16] for i in idx])

tensor([484709, 688230, 120568, 735235])


tensor([[39, 52,  1, 39,  1, 61, 39, 52, 58, 53, 52,  5, 57,  1, 40, 47],
        [57,  1, 46, 39, 52, 42, 57,  1, 40, 43, 45,  1, 51, 43, 56, 41],
        [26, 17, 26, 21, 33, 31, 10,  0, 13, 57,  1, 20, 43, 56, 41, 59],
        [52, 12,  1, 15, 39, 51, 47, 50, 50, 53,  1, 61, 47, 58, 46,  1]])

In [7]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    idx = torch.randint(0, len(data)-context_length-1, (batch_size,))
    X = torch.stack([data[i:i+context_length] for i in idx])
    y = torch.stack([data[i+1:i+1+context_length] for i in idx])
    X, y = X.to(device), y.to(device)
    return X, y
    


### GPT Engine

In [31]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.qurey = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.qurey(x)
        wei = k @ q.transpose(-2, -1) * C**0.5 # k(B,T,head_size) @ q.transpose(B, head_size, T) ==> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  
        wei = F.softmax(wei, dim=-1) #(B,T,T)
        self.dropout(wei)

        v = self.value(x) #(B, T, head_size)
        out = wei @ v    #(B,T,T) @ (B,T,head_size) ==> (B,T,head_size)

        return out 
    

class MultiHeadAttention(nn.Module):
    def __init__(self, n_head, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_head)]) 
        self.proj = nn.Linear(n_embd, n_embd, bias=False)  
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.concat([h(x) for h in self.heads], dim=-1)
        out =  self.dropout(self.proj(out))

        return out

class Feedforward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd, bias= False),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd, bias= False),
            nn.Dropout(dropout)
            )
        
    def forward(self, x):
        out = self.net(x)
        return out

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()  
        head_size = n_embd // n_head
        self.msa = MultiHeadAttention(n_head, head_size)
        self.ff = Feedforward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)

    def forward(self,x):
        x = x + self.msa(self.ln1(x))
        out = x + self.ff(self.ln2(x))
        return out
    

class gpt(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(context_length, n_embd)
        self.block = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.l_norm = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx, targets = None):
        B, T = idx.shape
        token_emb = self.token_embedding_table(idx) #(B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) #(T, n_embd)
        x = token_emb + pos_emb #(B, T, n_embd)
        x = self.block(x) #(B, T, n_embd)
        x = self.l_norm(x) #(B, T, n_embd)
        logits = self.lm_head(x) #(B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)    
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    def generate(self, idx, max_new_tokens):    
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -context_length:]
            logits, _ = self(idx_cond)  # (B, T, vocab_size)    
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx




In [32]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [34]:
model = gpt().to(device)

print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr= 1e-3)


# Training loop
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f'step: {iter} training loss {losses["train"]:.4f} validation loss {losses["val"]:.4f}')

    X, Y = get_batch('train')
    logits, loss = model(X, Y)

    # backward pass
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

      

0.208128 M parameters
step: 0 training loss 4.3803 validation loss 4.3780
step: 100 training loss 2.7114 validation loss 2.7251
step: 200 training loss 2.5637 validation loss 2.5629
step: 300 training loss 2.5049 validation loss 2.5297
step: 400 training loss 2.4646 validation loss 2.4797
step: 500 training loss 2.4330 validation loss 2.4479
step: 600 training loss 2.4075 validation loss 2.4195
step: 700 training loss 2.3832 validation loss 2.3870
step: 800 training loss 2.3442 validation loss 2.3642
step: 900 training loss 2.3323 validation loss 2.3573
step: 1000 training loss 2.2938 validation loss 2.3023
step: 1100 training loss 2.2672 validation loss 2.2887
step: 1200 training loss 2.2561 validation loss 2.2772
step: 1300 training loss 2.2284 validation loss 2.2481
step: 1400 training loss 2.2114 validation loss 2.2249
step: 1500 training loss 2.1781 validation loss 2.2028
step: 1600 training loss 2.1638 validation loss 2.2025
step: 1700 training loss 2.1463 validation loss 2.1875


KeyboardInterrupt: 